In [0]:
fetch_date = dbutils.widgets.get("fetch_date")
source_table = dbutils.widgets.get("source_table")
landing_table = dbutils.widgets.get("landing_table")
branch_table = dbutils.widgets.get("branch_table")
office_table = dbutils.widgets.get("office_table")
aradjustments_table = dbutils.widgets.get("aradjustments_table")
adjustmentcode_table = dbutils.widgets.get("adjustmentcode_table")
acthistory_table = dbutils.widgets.get("acthistory_table")

In [0]:
display(
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW transactions_src AS
SELECT 
    CAST(ReportingDate AS DATE) AS ReportingDate,
    CAST(PaymentID AS BIGINT) AS PaymentID,
    CAST(FacilityCode AS INT) AS FacilityCode,
    CAST(AcctNbr AS STRING) AS AcctNbr,
    CAST(TransDate AS INT) AS TransDate,
    CAST(EntryDate AS INT) AS EntryDate,
    CAST(TransAmt AS DOUBLE) AS TransAmt,
    CAST(TransCode AS STRING) AS TransCode,
    CAST(TransDesc AS STRING) AS TransDesc,
    CAST(TransType AS STRING) AS TransType,
    NULL AS InsCode,
    NULL AS Payor,
    NULL AS FinClass,
    NULL AS Coinsurance,
    NULL AS Deductible,
    NULL AS CoPay,
    NULL AS PatResp,
    NULL AS BatchID,
    CAST(SourceSystemKey AS INT) AS SourceSystemKey
FROM (
  WITH 
  transactions_cte AS (
    -- PAYMENTS
    SELECT 
      CAST('{fetch_date}' AS DATE) AS ReportingDate,
      CONCAT(date_format(act.ct_postdate, 'yyyyMMdd'), LPAD(act.ct_id, 6, '0')) AS PaymentID,
      CASE 
          WHEN b.branch_code RLIKE '[A-Za-z]' THEN ofc.OfficeNumber 
          ELSE b.branch_code 
      END AS FacilityCode,
      bi_h.i_id AS AcctNbr,
      date_format(act.ct_postdate, 'yyyyMMdd') AS TransDate,
      date_format(act.ct_postdate, 'yyyyMMdd') AS EntryDate,
      act.ct_initialamount AS TransAmt,
      'Other Payment' AS TransCode,
      act.ct_checknumber AS TransDesc,  -- Check number or EFT number
      'Payment' AS TransType,
      '6' AS SourceSystemKey
    FROM {source_table} bi_h
    JOIN {branch_table} b 
        ON bi_h.i_branchcode = b.branch_code
    LEFT JOIN {office_table} ofc
        ON ofc.OfficeAbbreviation = b.branch_code
    JOIN {acthistory_table} act
        ON act.ct_iid = bi_h.i_id
    WHERE bi_h.i_Balance <> 0
    UNION 
    -- ADJUSTMENTS
    SELECT DISTINCT
      CAST('{fetch_date}' AS DATE) AS ReportingDate,
      CONCAT(date_format(a.PostDate, 'yyyyMMdd'), LPAD(a.tc_id, 6, '0')) AS PaymentID,
      CASE 
          WHEN b.branch_code RLIKE '[A-Za-z]' THEN ofc.OfficeNumber 
          ELSE b.branch_code 
      END AS FacilityCode,
      bi_h.i_id AS AcctNbr,
      date_format(a.PostDate, 'yyyyMMdd') AS TransDate,
      date_format(a.PostDate, 'yyyyMMdd') AS EntryDate,
      a.adj_amt AS TransAmt,
      ac.AdjustmentCode AS TransCode,  -- Use this column
      NULL AS TransDesc,  -- Empty for adjustments per  expected data
      'Adjustment' AS TransType,
      '6' AS SourceSystemKey
    FROM {source_table} bi_h
    JOIN {branch_table} b 
        ON bi_h.i_branchcode = b.branch_code
    LEFT JOIN {office_table} ofc
        ON ofc.OfficeAbbreviation = b.branch_code
    JOIN {aradjustments_table} a 
        ON a.invnum = bi_h.i_id
    LEFT JOIN {adjustmentcode_table} ac
        ON ac.ac_id = a.tc_acid  -- Join on AdjustmentCodeID
    -- WHERE bi_h.i_id=294303658
    WHERE bi_h.i_Balance <> 0
  ),
  transactions_clean AS (
    SELECT *,
    row_number() OVER (PARTITION BY AcctNbr ORDER BY AcctNbr ) AS rn
    FROM transactions_cte
  )
  SELECT ReportingDate, PaymentID, FacilityCode, AcctNbr, TransDate, EntryDate, TransAmt, TransCode, TransDesc, TransType, SourceSystemKey
  FROM transactions_clean
  WHERE rn=1
)
""")
)

In [0]:
display(
spark.sql(f"""
MERGE INTO {landing_table} tgt
USING transactions_src src
    ON tgt.AcctNbr = src.AcctNbr
    AND tgt.ReportingDate = src.ReportingDate
    AND tgt.SourceSystemKey = 6

WHEN MATCHED THEN
UPDATE SET
    tgt.PaymentID = src.PaymentID,
    tgt.FacilityCode = src.FacilityCode,
    tgt.TransDate = src.TransDate,
    tgt.EntryDate = src.EntryDate,
    tgt.TransAmt = src.TransAmt,
    tgt.TransCode = src.TransCode,
    tgt.TransDesc = src.TransDesc,
    tgt.TransType = src.TransType,
    tgt.InsCode = src.InsCode,
    tgt.Payor = src.Payor,
    tgt.FinClass = src.FinClass,
    tgt.Coinsurance = src.Coinsurance,
    tgt.Deductible = src.Deductible,
    tgt.CoPay = src.CoPay,
    tgt.PatResp = src.PatResp,
    tgt.BatchID = src.BatchID,
    tgt.SourceSystemKey = src.SourceSystemKey,
    tgt._load_timestamp = current_timestamp()

WHEN NOT MATCHED THEN
INSERT (
    ReportingDate,
    PaymentID,
    FacilityCode,
    AcctNbr,
    TransDate,
    EntryDate,
    TransAmt,
    TransCode,
    TransDesc,
    TransType,
    InsCode,
    Payor,
    FinClass,
    Coinsurance,
    Deductible,
    CoPay,
    PatResp,
    BatchID,
    SourceSystemKey,
    _load_timestamp
)
VALUES (
    src.ReportingDate,
    src.PaymentID,
    src.FacilityCode,
    src.AcctNbr,
    src.TransDate,
    src.EntryDate,
    src.TransAmt,
    src.TransCode,
    src.TransDesc,
    src.TransType,
    src.InsCode,
    src.Payor,
    src.FinClass,
    src.Coinsurance,
    src.Deductible,
    src.CoPay,
    src.PatResp,
    src.BatchID,
    src.SourceSystemKey,
    current_timestamp()
)
""")
)